In [1]:
import numpy as np
import pandas as pd
import os
import xgboost as xgb

from sklearn.tree import DecisionTreeClassifier 
from sklearn.cross_validation import train_test_split

#记录程序运行时间
import time 
start_time = time.time()

/Users/Evan/anaconda3/lib/python3.5/site-packages/sklearn/cross_validation.py:44: DeprecationWarning: This module was deprecated in version 0.18 in favor of the model_selection module into which all the refactored classes and functions are moved. Also note that the interface of the new CV iterators are different from that of this module. This module will be removed in 0.20.
  "This module will be removed in 0.20.", DeprecationWarning)


In [2]:
os.chdir('/Users/Evan/Kaggle')

In [3]:
data_train = pd.read_csv('./Credit/data_train.csv')
data_test = pd.read_csv('./Credit/data_test.csv')

In [4]:
data_select_train = data_train.drop(['Unnamed: 0','timestamp_money','sum_total','mean_total','freq_total','sum_income','mean_income','freq_income','sum_spend','mean_spend','freq_spend'],axis = 1)
data_select_test = data_test.drop(['Unnamed: 0','timestamp_money','sum_total','mean_total','freq_total','sum_income','mean_income','freq_income','sum_spend','mean_spend','freq_spend'],axis = 1)

In [5]:
data_select_train.isnull().sum()/len(data_train)

ID                     0.000000
gender                 0.000000
career                 0.000000
education              0.000000
marriage               0.000000
hukou                  0.000000
Label                  0.000000
bill_credit            0.119811
credit_line            0.043564
cash_advance           0.043564
amount_transactions    0.043564
browser1               0.148680
browser2               0.148680
browser3               0.148680
browser4               0.148680
browser5               0.148680
browser6               0.148680
browser7               0.148680
browser8               0.148680
browser9               0.148680
browser10              0.148680
browser11              0.148680
dtype: float64

In [6]:
X = data_select_train.drop(['Label','ID'],axis =1)

In [7]:
y = data_select_train['Label']

In [8]:
data_select_test.columns

Index(['ID', 'gender', 'career', 'education', 'marriage', 'hukou', 'Label',
       'bill_credit', 'credit_line', 'cash_advance', 'amount_transactions',
       'browser1', 'browser2', 'browser3', 'browser4', 'browser5', 'browser6',
       'browser7', 'browser8', 'browser9', 'browser10', 'browser11'],
      dtype='object')

In [9]:
data_select_test = data_select_test.drop(['ID','Label'],axis =1)

## 缺失值处理

In [10]:
X = X.fillna(-999)
y = y.fillna(-999)
data_select_test = data_select_test.fillna(-999)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.3,random_state=161214)

In [12]:
xgb_val = xgb.DMatrix(X_test,label=y_test,missing = -999)
xgb_train = xgb.DMatrix(X_train,label=y_train,missing = -999)
xgb_test = xgb.DMatrix(data_select_test,missing = -999)

In [13]:
params={
'booster':'gbtree',
'objective': 'multi:softmax', #多分类的问题
'num_class':10, # 类别数，与 multisoftmax 并用
'gamma':0.1,  # 用于控制是否后剪枝的参数,越大越保守，一般0.1、0.2这样子。
'max_depth':12, # 构建树的深度，越大越容易过拟合
'lambda':2,  # 控制模型复杂度的权重值的L2正则化项参数，参数越大，模型越不容易过拟合。
'subsample':0.7, # 随机采样训练样本
'colsample_bytree':0.7, # 生成树时进行的列采样
'min_child_weight':3, 
# 这个参数默认是 1，是每个叶子里面 h 的和至少是多少，对正负样本不均衡时的 0-1 分类而言
#，假设 h 在 0.01 附近，min_child_weight 为 1 意味着叶子节点中最少需要包含 100 个样本。
#这个参数非常影响结果，控制叶子节点中二阶导的和的最小值，该参数值越小，越容易 overfitting。 
'silent':1 ,#设置成1则没有运行信息输出，最好是设置为0.
'eta': 0.007, # 如同学习率
'seed':1000,
'nthread':7,# cpu 线程数
#'eval_metric': 'auc'
}
plst = list(params.items())
num_rounds = 5000 # 迭代次数
watchlist = [(xgb_train, 'train'),(xgb_val, 'val')]

#训练模型并保存
# early_stopping_rounds 当设置的迭代次数较大时，early_stopping_rounds 可在一定的迭代次数内准确率没有提升就停止训练
model = xgb.train(plst, xgb_train, num_rounds, watchlist,early_stopping_rounds=100)
model.save_model('./xgb.model') # 用于存储训练出的模型
print("best best_ntree_limit",model.best_ntree_limit)

[0]	train-merror:0.128453	val-merror:0.128665
Multiple eval metrics have been passed: 'val-merror' will be used for early stopping.

Will train until val-merror hasn't improved in 100 rounds.
[1]	train-merror:0.127836	val-merror:0.128725
[2]	train-merror:0.128273	val-merror:0.128545
[3]	train-merror:0.127528	val-merror:0.128845
[4]	train-merror:0.127348	val-merror:0.128965
[5]	train-merror:0.127168	val-merror:0.128545
[6]	train-merror:0.126988	val-merror:0.128665
[7]	train-merror:0.127348	val-merror:0.128665
[8]	train-merror:0.127219	val-merror:0.128545
[9]	train-merror:0.127091	val-merror:0.128485
[10]	train-merror:0.127194	val-merror:0.128605
[11]	train-merror:0.127091	val-merror:0.128545
[12]	train-merror:0.127245	val-merror:0.128485
[13]	train-merror:0.127142	val-merror:0.128545
[14]	train-merror:0.127219	val-merror:0.128425
[15]	train-merror:0.127322	val-merror:0.128605
[16]	train-merror:0.127708	val-merror:0.128425
[17]	train-merror:0.12781	val-merror:0.128485
[18]	train-merror:0

In [14]:
preds = model.predict(xgb_test,ntree_limit=model.best_ntree_limit)

np.savetxt('xgb_submission.csv',np.c_[range(1,len(data_select_test)+1),preds],delimiter=',',header='ImageId,Label',comments='',fmt='%d')

#输出运行时长
cost_time = time.time()-start_time
print("xgboost success!",'\n',"cost time:",cost_time,"(s)......")

xgboost success! 
 cost time: 35.217331886291504 (s)......
